# Label quality

**Goal:** better labels and proof that our fakes are realistic.
1. Known-asteroid positions from **JPL Horizons** instead of SkyBoT.
2. **Relabel** every night with the better positions.
3. **Injection realism test:** can a classifier tell our fakes from real asteroids?
4. Re-measure v1 with the improved labels (a better ruler means re-measuring the old version too).

Read `docs/label_quality.md` alongside this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.label.refine_known_positions import refine_known_positions
from asthunt.label.load_known_asteroids import load_known_asteroids
from asthunt.dataset.relabel_night_detections import relabel_night_detections
from asthunt.evaluate.measure_injection_realism import measure_injection_realism

project = load_config("project"); paths = project["paths"]
acquire, inject = load_config("acquire"), load_config("inject")


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


all_nights = sum(acquire["nights"].values(), []) + acquire["holdout_nights"]
known_dir = paths["interim"] / "known_asteroids"

## 1. Ask Horizons (cached after the first run)

In [ ]:
for night in all_nights:
    exposures = pd.read_parquet(paths["processed"] / f"detections__{make_unit_tag(unit_of(night))}.parquet")["exposure"].unique()
    if all((known_dir / f"known_asteroids_horizons__{e}.csv").exists() for e in exposures):
        continue                                            # already refined
    science = [paths["raw"] / "ztf" / night / f"{e}_sciimg.fits" for e in exposures]
    print(night, refine_known_positions(science, paths["interim"]), "objects refined")
print("all nights refined")

## 2. Which positions agree with what ZTF actually saw?
For every known asteroid brighter than V = 21, the distance to the nearest real detection (SNR > 5) in its frame.

In [ ]:
offsets = {"skybot": [], "horizons": []}
for night in all_nights:
    detections = pd.read_parquet(paths["processed"] / f"detections__{make_unit_tag(unit_of(night))}.parquet")
    for exposure, rows in detections[detections["snr"] > 5].groupby("exposure"):
        tree = cKDTree(rows[["x", "y"]].to_numpy())
        tables = {"skybot": pd.read_csv(known_dir / f"known_asteroids__{exposure}.csv"),
                  "horizons": pd.read_csv(known_dir / f"known_asteroids_horizons__{exposure}.csv")}
        for source, table in tables.items():
            table = table[table["v_mag"] < 21]
            if len(table):
                distance, _ = tree.query(table[["x", "y"]].to_numpy())
                offsets[source] += list(distance[distance < 8])

bins = np.linspace(0, 8, 41)
for source, values in offsets.items():
    values = np.array(values)
    plt.hist(values, bins=bins, alpha=0.6, label=f"{source}: median {np.median(values):.2f} px")
    print(f"{source:8s}: {len(values)} matches, median {np.median(values):.2f} px, 90% within {np.percentile(values, 90):.2f} px")
plt.xlabel("distance from predicted position to detection (pixels)"); plt.legend(); plt.show()

SkyBoT's predictions sit ~2.7 px away; Horizons' ~0.3 px. With Horizons we can use a **2-px** match radius again
(`known_match_radius_pixels` in `configs/inject.yaml`): 4× less area, so 4× fewer chance coincidences.

## 3. Relabel every night (detections and stamps unchanged; only labels)

In [ ]:
rows = []
for night in all_nights:
    before = pd.read_parquet(paths["processed"] / f"detections__{make_unit_tag(unit_of(night))}.parquet")
    after = relabel_night_detections(unit_of(night), paths, inject)
    rows.append({"night": night, "known detections": int((after["known_asteroid"] != "").sum()),
                 "real labels": int(after["label"].sum()), "labels changed now": int((before["label"] != after["label"]).sum())})
pd.DataFrame(rows)

(If you run this notebook twice, "labels changed now" is 0 the second time: relabelling is **idempotent**. The first time, 12 of
~13,000 real labels changed: 5 doubtful matches removed, 7 real ones added.)

## 4. Are our fakes realistic? Adversarial validation
Class 1 = detections of real known asteroids, class 0 = fake detections **with the same magnitude distribution**.
A classifier cross-validated by night tries to tell them apart. Only training and validation nights are used.

In [ ]:
nights = acquire["nights"]["train"] + acquire["nights"]["validation"]
detections = pd.concat([pd.read_parquet(paths["processed"] / f"detections__{make_unit_tag(unit_of(n))}.parquet") for n in nights],
                       ignore_index=True)
auc, importances = measure_injection_realism(detections, np.arange(16, 22.5, 0.5), seed=project["seed"])
real_count = int((detections["known_asteroid"] != "").sum())
print(f"real known-asteroid detections: {real_count}")
print(f"realism AUC: {auc:.3f}   (0.5 = indistinguishable, 1.0 = trivially different)")
importances.round(3)

AUC ≈ 0.5: **the classifier can't tell our fakes from real asteroids.** That's the evidence behind "our fakes are realistic".
Caveat: 178 real detections is a small sample (uncertainty of a few hundredths), and it only covers the features tested.

Why do the raw medians differ (fake SNR ~13, real ~7)? Fakes are spread evenly over magnitudes 17-22, while real detected
asteroids are mostly faint. Comparing them without matching magnitudes would "prove" a difference that's just brightness.
**Always compare like with like.**

## 5. Re-measure v1 with the improved labels
Run `notebooks/scoreboard.ipynb` again. It replaces v1's rows (no duplicates), so every later version is compared with v1 on
the same, better labels.

In [ ]:
print(pd.read_csv(paths["reports"] / "scoreboard.csv").round(3).to_string())

## Exercises
1. Plot the Horizons offsets as arrows (dx, dy). Are they random in direction (as noise should be) or do they line up (a systematic error)?
2. Add `mag` itself to `REALISM_FEATURES` without magnitude matching. What AUC do you get, and why is it meaningless?
3. Which single feature would you check next to make the realism test stricter? (Hint: look at the stamps, not the catalogue numbers.)